In [ ]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
import warnings; warnings.simplefilter('ignore')
import os, sys, subprocess

def bootstrap_runtime(wheels_dir: str) -> None:
    """Install required packages from pre-extracted offline wheels."""
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-index',
        '--find-links', wheels_dir,
        'unsloth', 'trl', 'vllm', 'openai_harmony'
    ], check=True)

bootstrap_runtime(wheels_dir='/kaggle/input/datasets/chandan989/arc-aimo-utils/wheels')

In [ ]:
for env_key, env_val in {
    'TRANSFORMERS_NO_TF': '1',
    'TRANSFORMERS_NO_FLAX': '1',
    'CUDA_VISIBLE_DEVICES': '0',
    'TOKENIZERS_PARALLELISM': 'false',
    'TRITON_PTXAS_PATH': '/usr/local/cuda/bin/ptxas',
    'TIKTOKEN_ENCODINGS_BASE': '/kaggle/input/datasets/chandan989/arc-aimo-utils/tiktoken_encodings',
}.items():
    os.environ[env_key] = env_val

In [ ]:
import warnings; warnings.simplefilter('ignore')

import gc, re, math, time, queue, threading, contextlibimport gc, re, math, time, queue, threading, contextlib
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl
from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, load_harmony_encoding,
    SystemContent, ReasoningEffort, ToolNamespaceConfig,
    Author, Message, Role, TextContent, Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server
from dataclasses import dataclass, field
from typing import Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl
from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, load_harmony_encoding,
    SystemContent, ReasoningEffort, ToolNamespaceConfig,
    Author, Message, Role, TextContent, Conversation
)

from transformers import set_seed
import kaggle_evaluation.aimo_3_inference_server

## Hyperparameters
Ablation-validated settings — each parameter comment cites the regression observed when changed.

In [ ]:
class CFG:
    """Configuration — 44-notebook base + 5-component entropy."""

    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        "techniques, patterns, or analogous problems. Don't commit to one approach immediately.\\n"
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )

    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Brute-force verification for small cases\n\n'
        
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        
        'Remember: Code should support your mathematical reasoning, not replace it. '
        "Explain what you're computing and why before running code."
    )

    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'
        
        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds'
    )

    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/models/danielhanchen/gpt-oss-120b/transformers/default/1'

    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 1.0
    min_p = 0.02


set_seed(CFG.seed)

## Prompt Assembly

In [ ]:
class PromptAssembler:
    """Constructs the conversation seed from config directives."""

    @staticmethod
    def get_system_content(system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:
        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    @staticmethod
    def build_messages(
        system_prompt: str, user_prompt: str, tool_config: ToolNamespaceConfig
    ) -> list[Message]:
        system_content = PromptAssembler.get_system_content(system_prompt, tool_config)
        return [
            Message.from_role_and_content(Role.SYSTEM, system_content),
            Message.from_role_and_content(Role.USER, user_prompt),
        ]

## Kernel Orchestrator - Presistent Jupter Sandboxes

In [ ]:
_ANSI_RE = re.compile(r'\x1b\[[0-9;]*m')

_PREAMBLE = (
    'import math\n'
    'import numpy\n'
    'import sympy\n'
    'import itertools\n'
    'import collections\n'
    'import mpmath\n'
    'mpmath.mp.dps = 64\n'
)


class SandboxKernel:
    """Persistent IPython kernel for sandboxed code execution."""

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_ports(cls, n: int = 5) -> list[int]:
        with cls._port_lock:
            start = cls._next_port
            cls._next_port += n
            return list(range(start, start + n))

    def __init__(self, timeout: float):
        self._timeout = timeout
        self._active = False
        self._kc = None
        self._km = None

        ports = self._get_ports()
        env = os.environ.copy()
        env.update({
            'PYDEVD_DISABLE_FILE_VALIDATION': '1',
            'PYDEVD_WARN_EVALUATION_TIMEOUT': '0',
            'JUPYTER_PLATFORM_DIRS': '1',
            'PYTHONWARNINGS': 'ignore',
            'MPLBACKEND': 'Agg',
        })

        self._km = KernelManager()
        for attr, port in zip(
            ('shell_port', 'iopub_port', 'stdin_port', 'hb_port', 'control_port'), ports
        ):
            setattr(self._km, attr, port)

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])
        self._kc = self._km.blocking_client()
        self._kc.start_channels()
        self._kc.wait_for_ready(timeout=self._timeout)
        self._active = True
        self.execute(_PREAMBLE)

    def execute(self, code: str, timeout: float | None = None) -> str:
        kc = self._kc
        limit = timeout or self._timeout
        msg_id = kc.execute(code, store_history=True, allow_stdin=False, stop_on_error=False)

        stdout_parts, stderr_parts = [], []
        t0 = time.time()

        while True:
            if time.time() - t0 > limit:
                self._km.interrupt_kernel()
                return f'[ERROR] Execution timed out after {limit} seconds'
            try:
                msg = kc.get_iopub_msg(timeout=1.0)
            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            kind = msg.get('msg_type')
            body = msg.get('content', {})

            if kind == 'stream':
                bucket = stdout_parts if body.get('name') == 'stdout' else stderr_parts
                bucket.append(body.get('text', ''))
            elif kind == 'error':
                stderr_parts.append(self._fmt_tb(body.get('traceback', [])))
            elif kind in ('execute_result', 'display_data'):
                plain = body.get('data', {}).get('text/plain')
                if plain:
                    stdout_parts.append(plain if plain.endswith('\n') else f'{plain}\n')
            elif kind == 'status' and body.get('execution_state') == 'idle':
                break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)
        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr
        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    @staticmethod
    def _fmt_tb(frames: list[str]) -> str:
        out = []
        for f in frames:
            clean = _ANSI_RE.sub('', f)
            if 'File "' in clean and 'ipython-input' not in clean:
                continue
            out.append(clean)
        return ''.join(out)

    def reset(self):
        self.execute('%reset -f\n' + _PREAMBLE)

    def close(self):
        with contextlib.suppress(Exception):
            if self._kc:
                self._kc.stop_channels()
        if self._active and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)
            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def __del__(self):
        self.close()

## Code Runner - Tool Bridge Between LLM and Sandbox

In [ ]:
class CodeRunner:
    """Bridges LLM tool-call messages to the Jupyter sandbox."""

    def __init__(self, timeout: float, tool_prompt: str, sandbox=None):
        self._timeout = timeout
        self._tool_prompt = tool_prompt
        self._sandbox = sandbox
        self._exec_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_sandbox(self):
        if self._sandbox is None:
            with self._init_lock:
                if self._sandbox is None:
                    self._sandbox = SandboxKernel(timeout=self._timeout)

    @staticmethod
    def _auto_print(code: str) -> str:
        lines = code.strip().split('\n')
        if not lines:
            return code
        tail = lines[-1].strip()
        if 'print' in tail or 'import' in tail or not tail or tail.startswith('#'):
            return code
        lines[-1] = f'print({tail})'
        return '\n'.join(lines)

    @property
    def tool_config(self) -> ToolNamespaceConfig:
        return ToolNamespaceConfig(name='python', description=self._tool_prompt, tools=[])

    def _make_reply(self, text: str, channel: str | None = None) -> Message:
        msg = Message(
            author=Author(role=Role.TOOL, name='python'),
            content=[TextContent(text=text)],
        ).with_recipient('assistant')
        return msg.with_channel(channel) if channel else msg

    def handle(self, tool_msg: Message) -> list[Message]:
        self._ensure_sandbox()
        raw = tool_msg.content[0].text
        prepared = self._auto_print(raw)
        with self._exec_lock:
            try:
                result = self._sandbox.execute(prepared)
            except TimeoutError as exc:
                result = f'[ERROR] {exc}'
        return [self._make_reply(result, channel=tool_msg.channel)]

## Confidence Scorer - 5 Signal Composite Entropy
Ablation: +5-6 pts over naive mean entropy (V111 40 vs V136 35)

In [ ]:
def composite_confidence(token_logprob_dicts: list) -> float:
    """5-component composite confidence score. Lower = more confident.

    S1  Mean Shannon entropy              (weight 0.30)
    S2  Entropy standard deviation         (weight 0.20)
    S3  Exponential-decay positional mean  (weight 0.40) <- dominant
    S4  Fraction of high-uncertainty tokens (weight 0.90 = 0.30 x 3.0)
    S5  Longest low-uncertainty streak      (bonus -0.10 x ratio)
    """
    if not token_logprob_dicts:
        return float('inf')

    per_token_h = []
    for distrib in token_logprob_dicts:
        if not isinstance(distrib, dict) or not distrib:
            continue
        h = 0.0
        for lp in distrib.values():
            p = math.exp(lp)
            if p > 0:
                h -= p * math.log2(p)
        per_token_h.append(h)

    if not per_token_h:
        return float('inf')

    n = len(per_token_h)
    mu = sum(per_token_h) / n
    sigma = math.sqrt(sum((x - mu) ** 2 for x in per_token_h) / n)

    alpha = 0.995
    w = [alpha ** (n - 1 - i) for i in range(n)]
    w_sum = sum(w)
    recency = sum(wi * hi for wi, hi in zip(w, per_token_h)) / w_sum

    noisy_frac = sum(1 for x in per_token_h if x > 2.0) / n

    best_run = cur_run = 0
    for x in per_token_h:
        if x < 1.0:
            cur_run += 1
            best_run = max(best_run, cur_run)
        else:
            cur_run = 0
    streak_bonus = -0.1 * (best_run / n)

    return 0.3 * mu + 0.4 * recency + 0.2 * sigma + 0.9 * noisy_frac + streak_bonus


_BOXED_RE = re.compile(r'\\boxed\s*\{\s*([0-9,]+)\s*\}')
_VERBAL_RE = re.compile(r'final\s+answer\s+is\s*([0-9,]+)', re.IGNORECASE)


def extract_answer(text: str) -> int | None:
    """Pull a valid answer (0-99999) from \\boxed{} or 'final answer is' patterns."""
    for pattern in (_BOXED_RE, _VERBAL_RE):
        hits = pattern.findall(text)
        if hits:
            try:
                val = int(hits[-1].replace(',', ''))
                if 0 <= val <= 99999:
                    return val
            except ValueError:
                continue
    return None

## Inference Engine

In [ ]:
class InferenceEngine:
    """Top-level solver: boots vLLM, manages kernel pool, runs parallel rollouts."""

    def __init__(self, cfg, port: int = 8000):
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'

        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()

        self._preload_weights()
        self._server_proc = self._launch_vllm()

        self.client = OpenAI(
            base_url=self.base_url, api_key=self.api_key,
            timeout=self.cfg.session_timeout,
        )

        self._wait_for_server()
        self._init_kernels()

        self.clock_origin = time.time()
        self.problems_remaining = 50

    # ── startup ─────────────────────────────────────────────

    def _preload_weights(self):
        print(f'Loading weights from {self.cfg.model_path} ...')
        t0 = time.time()
        paths, nbytes = [], 0
        for dirpath, _, fnames in os.walk(self.cfg.model_path):
            for fn in fnames:
                fp = os.path.join(dirpath, fn)
                if os.path.isfile(fp):
                    paths.append(fp)
                    nbytes += os.path.getsize(fp)

        def _touch(p):
            with open(p, 'rb') as fh:
                while fh.read(1 << 30):
                    pass

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as pool:
            list(pool.map(_touch, paths))
        print(f'Cached {len(paths)} files ({nbytes / 1e9:.1f} GB) in {time.time() - t0:.1f}s\n')

    def _launch_vllm(self) -> subprocess.Popen:
        argv = [
            sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
            '--seed', str(self.cfg.seed),
            '--model', self.cfg.model_path,
            '--served-model-name', self.cfg.served_model_name,
            '--tensor-parallel-size', '1',
            '--max-num-seqs', str(self.cfg.batch_size),
            '--gpu-memory-utilization', str(self.cfg.gpu_memory_utilization),
            '--host', '0.0.0.0', '--port', str(self.port),
            '--dtype', self.cfg.dtype,
            '--kv-cache-dtype', self.cfg.kv_cache_dtype,
            '--max-model-len', str(self.cfg.context_tokens),
            '--stream-interval', str(self.cfg.stream_interval),
            '--async-scheduling', '--disable-log-stats',
            '--enable-prefix-caching',
        ]
        self._log_fh = open('vllm_server.log', 'w')
        return subprocess.Popen(
            argv, stdout=self._log_fh, stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    def _wait_for_server(self):
        print('Waiting for vLLM ...')
        t0 = time.time()
        for _ in range(self.cfg.server_timeout):
            rc = self._server_proc.poll()
            if rc is not None:
                self._log_fh.flush()
                raise RuntimeError(
                    f'vLLM exited with code {rc}.\n'
                    + open('vllm_server.log').read()
                )
            try:
                self.client.models.list()
                print(f'Ready in {time.time() - t0:.1f}s\n')
                return
            except Exception:
                time.sleep(1)
        raise RuntimeError('vLLM failed to start (timeout).')

    def _init_kernels(self):
        print(f'Provisioning {self.cfg.workers} Jupyter kernels ...')
        t0 = time.time()
        self._kernel_pool = queue.Queue()
        mk = lambda: SandboxKernel(timeout=self.cfg.jupyter_timeout)
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as pool:
            for k in pool.map(lambda _: mk(), range(self.cfg.workers)):
                self._kernel_pool.put(k)
        print(f'Ready in {time.time() - t0:.1f}s\n')

    # ── single rollout ────────────────────────────────────────

    def _run_attempt(self, problem, system_prompt, idx, halt, deadline):
        empty = {
            'Attempt': idx + 1, 'Answer': None, 'Python Calls': 0,
            'Python Errors': 0, 'Response Length': 0, 'Entropy': float('inf'),
        }
        if halt.is_set() or time.time() > deadline:
            return empty

        sandbox = None
        py_calls = py_errs = tok_count = 0
        answer = None
        lp_buf = []
        roll_seed = int(math.pow(self.cfg.seed + idx, 2))

        try:
            sandbox = self._kernel_pool.get(timeout=self.cfg.sandbox_timeout)
            runner = CodeRunner(
                timeout=self.cfg.jupyter_timeout,
                tool_prompt=self.cfg.tool_prompt,
                sandbox=sandbox,
            )
            msgs = PromptAssembler.build_messages(
                system_prompt, problem, runner.tool_config,
            )
            conv = Conversation.from_messages(msgs)

            for _ in range(self.cfg.turns):
                if halt.is_set() or time.time() > deadline:
                    break

                prompt_ids = self.encoding.render_conversation_for_completion(conv, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
                if max_tokens < self.cfg.buffer_tokens:
                    break

                stream = self.client.completions.create(
                    model=self.cfg.served_model_name,
                    temperature=self.cfg.temperature,
                    logprobs=self.cfg.top_logprobs,
                    max_tokens=max_tokens,
                    prompt=prompt_ids,
                    seed=roll_seed,
                    stream=True,
                    extra_body={
                        'min_p': self.cfg.min_p,
                        'stop_token_ids': self.stop_token_ids,
                        'return_token_ids': True,
                    },
                )

                try:
                    id_buf, txt_buf = [], []
                    for chunk in stream:
                        if halt.is_set() or time.time() > deadline:
                            break
                        ids = chunk.choices[0].token_ids
                        txt = chunk.choices[0].text
                        if ids:
                            id_buf.extend(ids)
                            tok_count += len(ids)
                            txt_buf.append(txt)
                            clp = chunk.choices[0].logprobs
                            if clp is not None and clp.top_logprobs:
                                lp_buf.extend(clp.top_logprobs)

                        if '}' in txt:
                            window = ''.join(txt_buf[-self.cfg.search_tokens:])
                            maybe = extract_answer(window)
                            if maybe is not None:
                                answer = maybe
                                break
                finally:
                    stream.close()

                if answer is not None:
                    break
                if not id_buf:
                    break

                new_msgs = self.encoding.parse_messages_from_completion_tokens(id_buf, Role.ASSISTANT)
                conv.messages.extend(new_msgs)
                tail = new_msgs[-1]

                if tail.channel == 'final':
                    answer = extract_answer(tail.content[0].text)
                    break

                if tail.recipient == 'python':
                    py_calls += 1
                    replies = runner.handle(tail)
                    out_text = replies[0].content[0].text
                    if any(s in out_text for s in ('[ERROR]', 'Traceback', 'Error:')):
                        py_errs += 1
                    conv.messages.extend(replies)

        except Exception:
            py_errs += 1
        finally:
            if sandbox is not None:
                sandbox.reset()
                self._kernel_pool.put(sandbox)

        return {
            'Attempt': idx + 1,
            'Response Length': tok_count,
            'Python Calls': py_calls,
            'Python Errors': py_errs,
            'Entropy': composite_confidence(lp_buf),
            'Answer': answer,
        }

    # ── answer selection (simple entropy-weighted, NO arbiter) ──

    def _select_answer(self, results: list) -> int:
        ans_weights = defaultdict(float)
        ans_votes = defaultdict(int)

        for r in results:
            ans = r['Answer']
            ent = r['Entropy']
            if ans is not None:
                ans_weights[ans] += 1.0 / max(ent, 1e-9)
                ans_votes[ans] += 1

        if not ans_weights:
            print('\nResult: 0\n')
            return 0

        board = sorted(
            [{'answer': a, 'votes': ans_votes[a], 'score': s}
             for a, s in ans_weights.items()],
            key=lambda row: row['score'], reverse=True,
        )
        display(pd.DataFrame(board).round({'score': 3}))

        pick = board[0]['answer']
        print(f'\nFinal Answer: {pick}\n')
        return pick

    # ── top-level problem solver ──────────────────────────────

    def solve(self, problem: str) -> int:
        print(f'\nProblem: {problem}\n')
        user_input = f'{problem} {self.cfg.preference_prompt}'

        elapsed = time.time() - self.clock_origin
        remaining = self.cfg.notebook_limit - elapsed
        reserve = max(0, self.problems_remaining - 1) * self.cfg.base_problem_timeout
        budget = min(
            max(remaining - reserve, self.cfg.base_problem_timeout),
            self.cfg.high_problem_timeout,
        )
        deadline = time.time() + budget
        print(f'Budget: {budget:.0f}s | Remaining: {self.problems_remaining}\n')

        records, answers = [], []
        halt = threading.Event()
        pool = ThreadPoolExecutor(max_workers=self.cfg.workers)

        try:
            futs = [
                pool.submit(
                    self._run_attempt,
                    user_input, self.cfg.system_prompt, i, halt, deadline,
                )
                for i in range(self.cfg.attempts)
            ]

            for f in as_completed(futs):
                try:
                    rec = f.result()
                    records.append(rec)
                    if rec['Answer'] is not None:
                        answers.append(rec['Answer'])
                    top = Counter(answers).most_common(1)
                    if top and top[0][1] >= self.cfg.early_stop:
                        halt.set()
                        for ft in futs:
                            ft.cancel()
                        break
                except Exception as exc:
                    print(f'Rollout error: {exc}')
        finally:
            halt.set()
            pool.shutdown(wait=True, cancel_futures=True)
            self.problems_remaining = max(0, self.problems_remaining - 1)

        if records:
            df = pd.DataFrame(records)
            df['Entropy'] = df['Entropy'].round(3)
            df['Answer'] = df['Answer'].astype('Int64')
            display(df[['Attempt', 'Response Length', 'Python Calls', 'Python Errors', 'Entropy', 'Answer']])

        if not answers:
            print('\nResult: 0\n')
            return 0

        return self._select_answer(records)

    # ── cleanup ───────────────────────────────────────────────

    def __del__(self):
        if hasattr(self, '_server_proc'):
            self._server_proc.terminate()
            self._server_proc.wait()
        if hasattr(self, '_log_fh'):
            self._log_fh.close()
        if hasattr(self, '_kernel_pool'):
            while not self._kernel_pool.empty():
                try:
                    self._kernel_pool.get_nowait().close()
                except Exception:
                    pass

## Initialize & Run

In [ ]:
engine = InferenceEngine(CFG)

In [ ]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    row_id = id_.item(0)
    q_text = question.item(0)
    gc.disable()
    result = engine.solve(q_text)
    gc.enable()
    gc.collect()
    return pl.DataFrame({'id': row_id, 'answer': result})

In [ ]:
harness = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    harness.serve()
else:
    harness.run_local_gateway(
        ('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )